# Proyek Pengembangan dan Pengoperasian Sistem Machine Learning
**Nama:** Muhammad Reza Pahlevi Harahap  
**Pipeline:** `reza_harahap-pipeline`

Notebook ini mendokumentasikan pembangunan dan eksekusi pipeline TFX untuk klasifikasi Breast Cancer Wisconsin. Semua komponen dijalankan menggunakan `BeamDagRunner` dan artifact disimpan pada direktori `reza_harahap-pipeline`.

## 1. Persiapan environment dan dataset
Dataset CSV dibaca oleh CsvExampleGen. Cell berikut memverifikasi versi TFX/TensorFlow, lokasi dataset, dan jumlah baris sebelum pipeline dibangun.

In [ ]:
from pathlib import Path
import pandas as pd
import tensorflow as tf
import tfx

DATA_PATH = Path("data/breast_cancer.csv")
df = pd.read_csv(DATA_PATH)
print("TFX version       :", tfx.__version__)
print("TensorFlow version:", tf.__version__)
print("Dataset           :", DATA_PATH)
print("Shape             :", df.shape)
print("Label counts:\n", df["label"].value_counts().sort_index())

## 2. Inisialisasi seluruh komponen TFX
Pipeline terdiri dari **CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator, Transform, Tuner, Trainer, Resolver, Evaluator, dan Pusher**. `create_pipeline()` pada `pipeline.py` menghubungkan output setiap komponen ke komponen berikutnya.

In [ ]:
from pipeline import create_pipeline, PIPELINE_ROOT, SERVING_MODEL_DIR
tfx_pipeline = create_pipeline()
print("Pipeline name:", tfx_pipeline.pipeline_info.pipeline_name)
print("Pipeline root:", PIPELINE_ROOT)
print("Serving model dir:", SERVING_MODEL_DIR)
print("\nKomponen TFX:")
for i, component in enumerate(tfx_pipeline.components, 1):
    print(f"{i:02d}. {component.id}")

## 3. Eksekusi pipeline dengan Apache Beam
`BeamDagRunner` menjalankan komponen secara berurutan. Cache diaktifkan sehingga eksekusi ulang dapat memakai artifact yang sudah valid. Output cell ini menjadi bukti bahwa pipeline benar-benar dieksekusi.

In [ ]:
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
BeamDagRunner().run(tfx_pipeline)
print("TFX PIPELINE EXECUTION COMPLETED")

## 4. Verifikasi artifact tiap komponen
Setelah pipeline selesai, direktori artifact diperiksa untuk memastikan hasil komponen bukan placeholder.

In [ ]:
from pathlib import Path
root = Path(PIPELINE_ROOT)
required = ["CsvExampleGen", "StatisticsGen", "SchemaGen", "ExampleValidator", "Transform", "Tuner", "Trainer", "Evaluator", "Pusher"]
print("Pipeline root exists:", root.exists())
print("Total artifact files:", sum(p.is_file() for p in root.rglob("*")))
for name in required:
    paths = [p for p in root.rglob("*") if name.lower() in str(p).lower()]
    print(f"{name:18s}: {len(paths)} path(s) ditemukan")
print("SavedModel:", list(Path(SERVING_MODEL_DIR).rglob("saved_model.pb")))

## 5. Ringkasan pipeline
- **ExampleGen** mengubah CSV menjadi TFRecord train/eval.
- **StatisticsGen** menghitung statistik fitur.
- **SchemaGen** menghasilkan schema protobuf text.
- **ExampleValidator** memeriksa anomali data.
- **Transform** menerapkan standardisasi z-score pada fitur numerik.
- **Tuner** mencari learning rate, hidden units, dan dropout.
- **Trainer** melatih binary classifier dan mengekspor SavedModel.
- **Evaluator** mengevaluasi BinaryAccuracy dan AUC dengan threshold 0.90.
- **Pusher** mendorong model yang diberkati ke `serving_model` untuk TensorFlow Serving.